In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [3]:
df = spark.read.json("data/transactions_10k.jsonl")

print("Liczba rekordów:", df.count())
df.printSchema()
df.show(10, truncate=False)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/notebooks/data/transactions_10k.jsonl. SQLSTATE: 42K03

In [4]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/notebooks/data/transactions_10k.jsonl. SQLSTATE: 42K03

In [5]:
df = spark.read.json("transactions_10k.jsonl")


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/notebooks/transactions_10k.jsonl. SQLSTATE: 42K03

In [6]:
import os
os.listdir()

['.git',
 '.ipynb_checkpoints',
 '.notremove',
 'consumer_anomaly.py',
 'consumer_count.py',
 'consumer_enrich.py',
 'consumer_filter.py',
 'lab2_spark.ipynb',
 'producer.py',
 'Untitled.ipynb',
 'Untitled1.ipynb',
 'Untitled2.ipynb',
 'Untitled3.ipynb',
 'Untitled4.ipynb',
 'Untitled5.ipynb']

In [7]:
import os
os.listdir("..")


['.bashrc',
 '.bash_logout',
 '.profile',
 '.gitconfig',
 '.npm',
 '.local',
 '.ipython',
 '.jupyter',
 'notebooks',
 '.conda',
 '.config',
 '.cache',
 '.wget-hsts',
 'work']

In [8]:
import os
os.listdir()

['.git',
 '.ipynb_checkpoints',
 '.notremove',
 'consumer_anomaly.py',
 'consumer_count.py',
 'consumer_enrich.py',
 'consumer_filter.py',
 'lab2_spark.ipynb',
 'producer.py',
 'Untitled.ipynb',
 'Untitled1.ipynb',
 'Untitled2.ipynb',
 'Untitled3.ipynb',
 'Untitled4.ipynb',
 'Untitled5.ipynb']

In [9]:
os.listdir("data")

FileNotFoundError: [Errno 2] No such file or directory: 'data'

In [10]:
import json
import random
from datetime import datetime, timedelta

stores = ["Warszawa", "Kraków", "Gdańsk", "Wrocław"]
categories = ["elektronika", "odzież", "żywność", "książki"]

start = datetime(2026, 5, 2, 8, 0, 0)

data = []

for i in range(10000):
    data.append({
        "tx_id": f"TX{i:05d}",
        "user_id": f"u{random.randint(1,20):02d}",
        "amount": round(random.uniform(5, 5000), 2),
        "store": random.choice(stores),
        "category": random.choice(categories),
        "timestamp": (start + timedelta(seconds=i)).strftime("%Y-%m-%d %H:%M:%S")
    })

with open("transactions_10k.jsonl", "w") as f:
    for row in data:
        f.write(json.dumps(row) + "\n")

print("OK - dane stworzone")

OK - dane stworzone


In [11]:
df = spark.read.json("transactions_10k.jsonl")

print(df.count())
df.printSchema()
df.show(10, truncate=False)

10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+-------+-----------+--------+-------------------+-------+-------+
|amount |category   |store   |timestamp          |tx_id  |user_id|
+-------+-----------+--------+-------------------+-------+-------+
|2534.48|książki    |Warszawa|2026-05-02 08:00:00|TX00000|u02    |
|1230.73|książki    |Gdańsk  |2026-05-02 08:00:01|TX00001|u13    |
|1715.64|książki    |Wrocław |2026-05-02 08:00:02|TX00002|u01    |
|442.61 |książki    |Kraków  |2026-05-02 08:00:03|TX00003|u11    |
|3914.56|książki    |Warszawa|2026-05-02 08:00:04|TX00004|u20    |
|3428.4 |odzież     |Gdańsk  |2026-05-02 08:00:05|TX00005|u07    |
|2539.14|żywność    |Kraków  |2026-05-02 08:00:06|TX00006|u16    |
|3698.34|odzież     |Warszawa|2026-05-02 08:00:07|TX00007|u10    |
|4762.2 |żywność   

In [12]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")
)

df.printSchema()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [13]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2511|6291300.55|     2505.5|
|  Kraków|     2457|6148350.91|    2502.38|
|Warszawa|     2510|6345489.12|    2528.08|
| Wrocław|     2522|6275541.97|    2488.32|
+--------+---------+----------+-----------+



In [14]:
from pyspark.sql.functions import min as _min, max as _max, sum as _sum, count

df.groupBy("category").agg(
    count("tx_id").alias("liczba_tx"),
    _sum("amount").alias("suma_PLN"),
    _min("amount").alias("min_PLN"),
    _max("amount").alias("max_PLN")
).orderBy("category").show()

+-----------+---------+-----------------+-------+-------+
|   category|liczba_tx|         suma_PLN|min_PLN|max_PLN|
+-----------+---------+-----------------+-------+-------+
|elektronika|     2527|6143710.069999991|   6.82|4994.61|
|    książki|     2499|6319673.609999992|   5.07| 4998.8|
|     odzież|     2467|6282106.110000001|   7.34|4998.62|
|    żywność|     2507|6315192.759999996|   5.81|4999.15|
+-----------+---------+-----------------+-------+-------+



In [15]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-05-02 08:00:00, 2026-05-02 09:00:00}|3600     |9079336.24|
|{2026-05-02 09:00:00, 2026-05-02 10:00:00}|3600     |9000737.48|
|{2026-05-02 10:00:00, 2026-05-02 11:00:00}|2800     |6980608.83|
+------------------------------------------+---------+----------+



In [16]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-05-02 08:00:00|2026-05-02 09:00:00|3600     |9079336.24|
|2026-05-02 09:00:00|2026-05-02 10:00:00|3600     |9000737.48|
|2026-05-02 10:00:00|2026-05-02 11:00:00|2800     |6980608.83|
+-------------------+-------------------+---------+----------+



In [17]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round

result_30min = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window", "store")
)

result_30min.show(truncate=False)

+------------------------------------------+--------+---------+----------+
|window                                    |store   |liczba_tx|suma_PLN  |
+------------------------------------------+--------+---------+----------+
|{2026-05-02 08:00:00, 2026-05-02 08:30:00}|Gdańsk  |458      |1163411.92|
|{2026-05-02 08:00:00, 2026-05-02 08:30:00}|Kraków  |436      |1098587.02|
|{2026-05-02 08:00:00, 2026-05-02 08:30:00}|Warszawa|452      |1140825.45|
|{2026-05-02 08:00:00, 2026-05-02 08:30:00}|Wrocław |454      |1212542.37|
|{2026-05-02 08:30:00, 2026-05-02 09:00:00}|Gdańsk  |465      |1157571.75|
|{2026-05-02 08:30:00, 2026-05-02 09:00:00}|Kraków  |436      |1102507.41|
|{2026-05-02 08:30:00, 2026-05-02 09:00:00}|Warszawa|456      |1173233.02|
|{2026-05-02 08:30:00, 2026-05-02 09:00:00}|Wrocław |443      |1030657.3 |
|{2026-05-02 09:00:00, 2026-05-02 09:30:00}|Gdańsk  |414      |1031279.8 |
|{2026-05-02 09:00:00, 2026-05-02 09:30:00}|Kraków  |446      |1133629.23|
|{2026-05-02 09:00:00, 20

In [18]:
from pyspark.sql.functions import window, sum as _sum, desc

krakow_hourly = (
    df.filter(df.store == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _sum("amount").alias("suma_PLN")
    )
    .orderBy(desc("suma_PLN"))
)

krakow_hourly.show(truncate=False)

+------------------------------------------+------------------+
|window                                    |suma_PLN          |
+------------------------------------------+------------------+
|{2026-05-02 09:00:00, 2026-05-02 10:00:00}|2327442.8000000003|
|{2026-05-02 08:00:00, 2026-05-02 09:00:00}|2201094.430000003 |
|{2026-05-02 10:00:00, 2026-05-02 11:00:00}|1619813.6800000006|
+------------------------------------------+------------------+



In [19]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-05-02 07:30:00|2026-05-02 08:30:00|1800     |4615366.76|
|2026-05-02 08:00:00|2026-05-02 09:00:00|3600     |9079336.24|
|2026-05-02 08:30:00|2026-05-02 09:30:00|3600     |8958541.59|
|2026-05-02 09:00:00|2026-05-02 10:00:00|3600     |9000737.48|
|2026-05-02 09:30:00|2026-05-02 10:30:00|3600     |8965346.32|
|2026-05-02 10:00:00|2026-05-02 11:00:00|2800     |6980608.83|
|2026-05-02 10:30:00|2026-05-02 11:30:00|1000     |2521427.88|
+-------------------+-------------------+---------+----------+



In [20]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ:

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [21]:
#Sliding ma więcej wierszy, ponieważ okna czasowe nakładają się na siebie i przesuwają co 30 minut. W efekcie te same dane są liczone wielokrotnie w różnych oknach (np. 09:00–10:00 i 09:30–10:30), co zwiększa liczbę wynikowych rekordów w porównaniu do tumbling window, gdzie każde zdarzenie należy tylko do jednego okna.

In [22]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ:ODPOWIEDŹ: W oknie 09:00–10:00 znajduje się 3600 transakcji.

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ:ODPOWIEDŹ:groupBy("store") wykonuje agregację dla całego zbioru danych bez uwzględnienia czasu, czyli daje jedną statystykę na sklep; groupBy(window(...), "store") dzieli dane na przedziały czasowe i dla każdego okna liczy statystyki osobno dla każdego sklepu, co pozwala analizować zmiany w czasie.

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ:ODPOWIEDŹ: Transakcja z godziny 09:30 należy do kilku nakładających się okien sliding window. W przypadku okna 1h przesuwanego co 30 minut, transakcja 09:30 znajduje się w dwóch oknach: - 09:00–10:00 - 09:30–10:30

In [23]:
from pyspark.sql.functions import window, avg, asc

gdansk_lowest_avg = (
    df.filter(df.store == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(avg("amount").alias("avg_amount"))
    .orderBy(asc("avg_amount"))
)

gdansk_lowest_avg.show(truncate=False)

+------------------------------------------+------------------+
|window                                    |avg_amount        |
+------------------------------------------+------------------+
|{2026-05-02 09:00:00, 2026-05-02 10:00:00}|2498.873404507711 |
|{2026-05-02 10:00:00, 2026-05-02 11:00:00}|2501.700134228186 |
|{2026-05-02 08:00:00, 2026-05-02 09:00:00}|2514.6085265438787|
+------------------------------------------+------------------+



In [24]:
#Najniższa średnia kwota transakcji w sklepie Gdańsk występuje w godzinie 09:00–10:00, ponieważ w tym oknie wartość avg_amount jest najniższa (2498.87).

In [25]:
from pyspark.sql.functions import count

cat_0930 = (
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(count("tx_id").alias("liczba_tx"))
    .filter("window.start = '2026-05-02 09:00:00'")
)

cat_0930.show()

+--------------------+-----------+---------+
|              window|   category|liczba_tx|
+--------------------+-----------+---------+
|{2026-05-02 09:00...|elektronika|      439|
|{2026-05-02 09:00...|    książki|      476|
|{2026-05-02 09:00...|     odzież|      459|
|{2026-05-02 09:00...|    żywność|      426|
+--------------------+-----------+---------+



In [26]:
#W oknie 09:00–09:30 liczba transakcji różni się w zależności od kategorii. Najwięcej transakcji wystąpiło w kategorii "książki" (476), a najmniej w kategorii "żywność" (426).

In [27]:
from pyspark.sql.functions import count, desc

peak_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy(desc("liczba_tx"))
)

peak_15min.show(truncate=False)

+------------------------------------------+---------+
|window                                    |liczba_tx|
+------------------------------------------+---------+
|{2026-05-02 09:30:00, 2026-05-02 09:45:00}|900      |
|{2026-05-02 09:45:00, 2026-05-02 10:00:00}|900      |
|{2026-05-02 08:00:00, 2026-05-02 08:15:00}|900      |
|{2026-05-02 08:15:00, 2026-05-02 08:30:00}|900      |
|{2026-05-02 10:15:00, 2026-05-02 10:30:00}|900      |
|{2026-05-02 08:45:00, 2026-05-02 09:00:00}|900      |
|{2026-05-02 10:00:00, 2026-05-02 10:15:00}|900      |
|{2026-05-02 10:30:00, 2026-05-02 10:45:00}|900      |
|{2026-05-02 08:30:00, 2026-05-02 08:45:00}|900      |
|{2026-05-02 09:15:00, 2026-05-02 09:30:00}|900      |
|{2026-05-02 09:00:00, 2026-05-02 09:15:00}|900      |
|{2026-05-02 10:45:00, 2026-05-02 11:00:00}|100      |
+------------------------------------------+---------+



In [28]:
#Największa liczba transakcji w oknach 15-minutowych wynosi 900 i występuje w wielu ćwierćgodzinach w ciągu dnia (m.in. 08:00–08:15, 09:00–09:15, 09:15–09:30 itd.). Oznacza to, że szczyt aktywności transakcyjnej jest rozłożony równomiernie w czasie, a nie skupiony w jednym pojedynczym oknie.